# 02 · RL Training + Backtest Evaluation

**Purpose**: Train a DQN agent on the merged HMM + news state, evaluate against baselines,
and report per-regime attribution and tail-risk metrics.

**Prerequisites**: Run `01_data_regimes.ipynb` first to generate
`output/full_pipeline/model_state_weekly_hmm_news.csv`.

**Key options** (configure in the cell below):
| Flag | Effect |
|---|---|
| `FAST_MODE = True` | Quick smoke-test (4 k steps, seq=4). Set `False` for production (30 k steps, seq=12). |
| `USE_MULTI_SEED = True` | Train 3 seeds, build ensemble majority-vote policy. |
| `USE_ATTENTION = True` | Use SB3-compatible LSTM+attention feature extractor. |
| `REWARD_MODE` | `"net_return"` (default) or `"dsr"` (Differential Sharpe Ratio). |


## 1 · Imports & Setup

In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_ROOT = None
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "full_pipeline").exists() and (_c / "scripts").exists():
        REPO_ROOT = _c
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root.")

PIPELINE_ROOT = REPO_ROOT / "full_pipeline"
for _p in (str(REPO_ROOT), str(PIPELINE_ROOT)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from evaluation import (
    BacktestEngine,
    EvaluationConfig,
    EnsembleActionPolicy,
    all_baseline_policies,
    bootstrap_metric_table,
    compare_strategies_bootstrap,
    default_action_space,
    load_default_dataset,
    per_regime_metrics,
    plot_equity_curves,
    PrecomputedActionPolicy,
    summary_table,
)
from ml.training_utils import (
    evaluate_episode,
    train_dqn_finrl,
    train_dqn_multi_seed,
)
from _pipeline_utils import (
    OUTPUT_DIR,
    make_rl_env,
    prepare_rl_inputs,
    rollout_agent_on_split,
    save_action_frame,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
print("REPO_ROOT:", REPO_ROOT)


## 2 · Training Configuration

In [ ]:
# ── Toggle these before running ──────────────────────────────────────────────
FAST_MODE         = True    # False → production run (30 k steps, seq_len=12)
USE_MULTI_SEED    = False   # True  → 3 seeds, ensemble majority-vote policy
USE_ATTENTION     = True    # True  → SB3 LSTM+attention extractor
REWARD_MODE       = "net_return"   # "net_return" or "dsr"
SEEDS             = [7, 21, 42]    # seeds used when USE_MULTI_SEED=True
# ─────────────────────────────────────────────────────────────────────────────

TOTAL_TIMESTEPS       = 4_000  if FAST_MODE else 30_000
EVAL_FREQ             = 500    if FAST_MODE else 2_000
EARLY_STOPPING_PATIENCE = 6   if FAST_MODE else 10
BUFFER_SIZE           = 5_000  if FAST_MODE else 10_000
SEQ_LEN               = 4     if FAST_MODE else 12

EVAL_CONFIG = EvaluationConfig(
    transaction_cost=0.001,
    risk_penalty=0.05,
    risk_window=12,
)

print(
    f"FAST_MODE={FAST_MODE} | timesteps={TOTAL_TIMESTEPS} | seq_len={SEQ_LEN}\n"
    f"USE_MULTI_SEED={USE_MULTI_SEED} | USE_ATTENTION={USE_ATTENTION} | "
    f"REWARD_MODE={REWARD_MODE!r}"
)


## 3 · Load Merged RL State

Evaluation splits:
- **train**: up to 2020-12-31
- **validation**: 2021-01-01 – 2022-12-30
- **locked_test**: after 2022-12-30 (held-out — do not tune on this)


In [ ]:
prepared = prepare_rl_inputs()
dataset  = prepared["dataset"]
frame    = prepared["frame"]

display(dataset.describe_splits())
display(dataset.describe_feature_blocks())

print("Feature columns:", len(prepared["feature_cols"]))
print("Posterior columns:", prepared["posterior_cols"])
display(frame[["week_end", "eval_split", *prepared["posterior_cols"][:2]]].head(3))


In [ ]:
# Feature scaling sanity check (train set only — validation/test must NOT influence the scaler).
train_mask   = frame["eval_split"] == "train"
scaled_train = prepared["scaled_features"].loc[train_mask, prepared["feature_cols"]]

display(pd.DataFrame({
    "train_mean_abs_max": [float(np.abs(scaled_train.mean()).max())],
    "train_std_min":      [float(scaled_train.std(ddof=0).min())],
    "train_std_max":      [float(scaled_train.std(ddof=0).max())],
}))


## 4 · Build Gymnasium Environments

In [ ]:
def _make_env(split):
    return make_rl_env(
        prepared, split=split, seq_len=SEQ_LEN,
        config=EVAL_CONFIG,
        reward_mode=REWARD_MODE,
        turnover_penalty=0.1,   # quadratic turnover penalty (0 = off)
        reward_clip=0.10,
    )

train_env = _make_env("train")
val_env   = _make_env("validation")
test_env  = _make_env("locked_test")

print("Action names :", train_env.ACTION_NAMES)
print("Obs space    :", train_env.observation_space)
print("Action space :", train_env.action_space)


## 5 · Train DQN Agent

Two training paths are available:
- **Single seed** (`USE_MULTI_SEED=False`): standard SB3 DQN, fast to iterate.
- **Multi-seed** (`USE_MULTI_SEED=True`): trains N independent seeds and builds an
  `EnsembleActionPolicy` (majority vote). Reduces seed variance at the cost of N× training time.


In [ ]:
TRAIN_KWARGS = dict(
    total_timesteps           = TOTAL_TIMESTEPS,
    eval_freq                 = EVAL_FREQ,
    early_stopping_patience   = EARLY_STOPPING_PATIENCE,
    learning_rate             = 1e-4,
    exploration_fraction      = 0.15,
    exploration_final_eps     = 0.05,
    target_update_interval    = 1_000,
    buffer_size               = BUFFER_SIZE,
    batch_size                = 32,
    device                    = "auto",
    use_attention_extractor   = USE_ATTENTION,
    verbose                   = 0,
)

if USE_MULTI_SEED:
    multi = train_dqn_multi_seed(
        train_env_factory = _make_env("train"),
        val_env_factory   = _make_env("validation"),
        seeds             = SEEDS,
        **TRAIN_KWARGS,
    )
    agents = multi["agents"]
    print(f"Multi-seed: mean_best_val={multi['mean_best_val']:.4f} "
          f"± {multi['std_best_val']:.4f}  (seeds {SEEDS})")
else:
    result = train_dqn_finrl(
        train_env = _make_env("train"),
        val_env   = _make_env("validation"),
        seed      = SEEDS[0],
        **TRAIN_KWARGS,
    )
    agents = [result["agent"]]
    print(f"Single-seed best val reward: {result['best_val_reward']:.4f}")
    display(pd.DataFrame(result["val_history"]).tail())


## 6 · Quick Agent Evaluation

In [ ]:
# Evaluate primary agent on validation and locked-test splits.
primary_agent = agents[0]

val_eval  = evaluate_episode(primary_agent, _make_env("validation"),  deterministic=True)
test_eval = evaluate_episode(primary_agent, _make_env("locked_test"), deterministic=True)

display(pd.DataFrame([
    {"split": "validation",  **val_eval},
    {"split": "locked_test", **test_eval},
])[["split", "reward", "length", "cumulative_return", "sharpe_ratio", "max_drawdown"]])


## 7 · Export Action Files

In [ ]:
val_env_export  = _make_env("validation")
test_env_export = _make_env("locked_test")

if USE_MULTI_SEED and len(agents) > 1:
    # Build ensemble actions from each seed, then save the majority-vote sequence.
    seed_val_actions  = [rollout_agent_on_split(a, _make_env("validation"),  frame, "validation")["action_id"]
                         for a in agents]
    seed_test_actions = [rollout_agent_on_split(a, _make_env("locked_test"), frame, "locked_test")["action_id"]
                         for a in agents]
    ensemble_val  = EnsembleActionPolicy(seed_val_actions,  name="ensemble_rl")
    ensemble_test = EnsembleActionPolicy(seed_test_actions, name="ensemble_rl")

    val_actions  = rollout_agent_on_split(primary_agent, val_env_export,  frame, "validation")
    test_actions = rollout_agent_on_split(primary_agent, test_env_export, frame, "locked_test")

    # Overwrite action_id column with ensemble votes.
    val_actions["action_id"]  = ensemble_val._voted_actions
    test_actions["action_id"] = ensemble_test._voted_actions
    val_actions["action_name"]  = [train_env.ACTION_NAMES[a] for a in val_actions["action_id"]]
    test_actions["action_name"] = [train_env.ACTION_NAMES[a] for a in test_actions["action_id"]]
else:
    val_actions  = rollout_agent_on_split(primary_agent, val_env_export,  frame, "validation")
    test_actions = rollout_agent_on_split(primary_agent, test_env_export, frame, "locked_test")

val_path  = save_action_frame(val_actions,  OUTPUT_DIR / "rl_validation_actions.csv")
test_path = save_action_frame(test_actions, OUTPUT_DIR / "rl_locked_test_actions.csv")

print("Saved:")
print(" ", val_path.relative_to(REPO_ROOT))
print(" ", test_path.relative_to(REPO_ROOT))
display(val_actions.head(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
val_actions["action_name"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Validation — Action Mix")
test_actions["action_name"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], title="Locked-Test — Action Mix")
axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()


## 8 · Backtest Evaluation vs Baselines

Evaluates all canonical baselines against the saved RL action file on both splits.


In [ ]:
state_path   = OUTPUT_DIR / "model_state_weekly_hmm_news.csv"
dataset_eval = load_default_dataset(state_path)
action_space = default_action_space()
engine       = BacktestEngine(
    dataset=dataset_eval,
    action_space=action_space,
    config=EvaluationConfig(transaction_cost=0.001, risk_penalty=0.05, risk_window=12),
)

include_blocks = ("price", "macro", "regime", "text")
baselines = all_baseline_policies(action_space)

val_results  = engine.run_many(baselines, split="validation",  include_blocks=include_blocks)
test_results = engine.run_many(baselines, split="locked_test", include_blocks=include_blocks)

# Load saved RL actions and evaluate via precomputed-action policy.
val_csv  = pd.read_csv(OUTPUT_DIR / "rl_validation_actions.csv",  parse_dates=["week_end"])
test_csv = pd.read_csv(OUTPUT_DIR / "rl_locked_test_actions.csv", parse_dates=["week_end"])

rl_val  = engine.evaluate_precomputed_actions(
    action_ids=val_csv["action_id"],  split="validation",  include_blocks=include_blocks, name="rl_dqn")
rl_test = engine.evaluate_precomputed_actions(
    action_ids=test_csv["action_id"], split="locked_test", include_blocks=include_blocks, name="rl_dqn")

print("Validation — all strategies")
display(summary_table(val_results  + [rl_val]))
print("\nLocked-test — all strategies")
display(summary_table(test_results + [rl_test]))


In [ ]:
# Bootstrap confidence intervals on Sharpe (locked-test only — 500 replications).
bootstrap_metric_table(test_results + [rl_test], metric="sharpe_ratio", n_boot=500, seed=7)


In [ ]:
plot_equity_curves(test_results + [rl_test], title="Locked-Test Equity Curves: Baselines vs RL")
plt.show()


## 9 · Per-Regime Attribution & Tail Risk

Splits the locked-test history by HMM regime label and reports performance within each regime.
Useful for diagnosing whether RL gains in one regime offset losses in another.


In [ ]:
# Per-regime metrics for RL agent on locked-test.
if rl_test.history is not None and "regime_label" in rl_test.history.columns:
    print("=== RL agent — per-regime metrics (locked-test) ===")
    display(per_regime_metrics(rl_test.history))
else:
    print("regime_label column not present in history — re-run backtest engine after updating to v2 backtest.")


In [ ]:
# Bootstrap Sharpe test: RL vs buy-and-hold SPY on locked-test.
spy_result = next((r for r in test_results if r.name == "buy_hold_spy"), None)
if spy_result is not None and rl_test.history is not None and spy_result.history is not None:
    boot = compare_strategies_bootstrap(
        rl_test.history["net_return"].values,
        spy_result.history["net_return"].values,
    )
    print("Bootstrap Sharpe test: RL vs SPY buy-and-hold (locked-test)")
    display(pd.DataFrame([boot]).T.rename(columns={0: "value"}))


In [ ]:
# Tail risk summary for all locked-test results.
from evaluation.metrics import compute_portfolio_metrics

tail_rows = []
for result in test_results + [rl_test]:
    if result.history is None:
        continue
    m = compute_portfolio_metrics(result.history)
    tail_rows.append({
        "strategy":         result.name,
        "sharpe":           round(m.get("sharpe_ratio", float("nan")), 3),
        "cvar_95":          round(m.get("cvar_95", float("nan")), 4),
        "max_drawdown":     round(m.get("max_drawdown", float("nan")), 4),
        "ulcer_index":      round(m.get("ulcer_index", float("nan")), 4),
        "martin_ratio":     round(m.get("martin_ratio", float("nan")), 3),
        "tail_ratio":       round(m.get("tail_ratio", float("nan")), 3),
        "downside_dev":     round(m.get("downside_deviation", float("nan")), 4),
    })

print("Tail risk table — locked-test")
display(pd.DataFrame(tail_rows).set_index("strategy").sort_values("sharpe", ascending=False))
